## IDN

In [1]:
import os, re, json, math, pdfplumber, torch, csv
from pathlib import Path
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "facebook/bart-large-mnli"
device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
model      = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(device)
label2id   = {v.lower(): k for k, v in model.config.id2label.items()}


In [ ]:
import re
import csv
import json
import torch
import pdfplumber
from tqdm import tqdm
from pathlib import Path

def extract_pdf(path: str) -> str:
    text = []
    with pdfplumber.open(path) as pdf:
        for p in pdf.pages:
            page = p.extract_text() or ""
            text.append(page)
    return re.sub(r"\s+", " ", " ".join(text)).strip()


def chunk_text(txt: str, max_tokens: int = 256, overlap: int = 64):
    words = txt.split()
    step = max_tokens - overlap
    for i in range(0, len(words), step):
        yield " ".join(words[i:i + max_tokens])


@torch.inference_mode()
def nli_score(premise_chunks, hypothesis: str):
    max_scores = {"entailment": 0.0, "neutral": 0.0, "contradiction": 0.0}
    for prem in premise_chunks:
        enc = tokenizer(prem, hypothesis, truncation=True, return_tensors="pt").to(device)
        logits = model(**enc).logits
        probs = torch.softmax(logits, -1)[0]
        max_scores["entailment"] = max(max_scores["entailment"], probs[label2id["entailment"]].item())
        max_scores["neutral"] = max(max_scores["neutral"], probs[label2id["neutral"]].item())
        max_scores["contradiction"] = max(max_scores["contradiction"], probs[label2id["contradiction"]].item())
    return max_scores


def extract_range(filename):
    match = re.search(r"(\d{1,3}-\d{1,3})", filename)
    return match.group(1) if match else None


def process_all(doc_dir: str, json_dir: str, output_score: str):
    doc_dir = Path(doc_dir)
    json_dir = Path(json_dir)
    output_score = Path(output_score)
    output_score.mkdir(parents=True, exist_ok=True)

    # Loop per subdirektori (contoh D1-xxx)
    for doc_subdir in sorted(doc_dir.iterdir()):
        if not doc_subdir.is_dir():
            continue

        dir_prefix = doc_subdir.name.split("-")[0]  # contoh: D3
        json_subdir = next((d for d in json_dir.iterdir() if d.name.startswith(dir_prefix)), None)

        if not json_subdir:
            print(f"Skip: Tidak ditemukan pasangan untuk {doc_subdir.name}!!")
            continue

        print(f"\n=== Memproses direktori: {doc_subdir.name} ===")

        results = []
        pdf_files = list(doc_subdir.glob("*.pdf"))
        json_files = list(json_subdir.glob("*.json"))

        json_map = {extract_range(f.name): f for f in json_files if extract_range(f.name)}

        for pdf_path in tqdm(pdf_files, desc=f"Proses {doc_subdir.name}"):
            range_key = extract_range(pdf_path.name)
            if not range_key or range_key not in json_map:
                print(f" Skip: {pdf_path.name} tidak ada pasangan JSON yang cocok")
                continue

            json_path = json_map[range_key]
            try:
                premise = extract_pdf(str(pdf_path))
                premise_chunks = list(chunk_text(premise))

                qna_data = json.loads(json_path.read_text(encoding="utf-8"))
                source = "grok" if "grok" in json_path.name.lower() else "gpt"

                for item in qna_data:
                    question = item.get("Question", "").strip()
                    answer = item.get("Answer", "").strip()
                    hypothesis = answer or f"{question} {answer}".strip()
                    scores = nli_score(premise_chunks, hypothesis)

                    results.append({
                        "file": pdf_path.stem,
                        "generator": source,
                        "question": question,
                        "answer": answer,
                        "score_entailment": round(scores["entailment"], 4),
                        "score_neutral": round(scores["neutral"], 4),
                        "score_contradiction": round(scores["contradiction"], 4),
                        "score_final": round(scores["entailment"] - scores["contradiction"], 4),
                    })
            except Exception as e:
                print(f"Gagal memproses {pdf_path.name}: {e}")

        output_csv = output_score / f"{doc_subdir.name}.csv"
        with open(output_csv, "w", encoding="utf-8", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=[
                "file", "generator", "question", "answer",
                "score_entailment", "score_neutral", "score_contradiction", "score_final"
            ], delimiter=";")
            writer.writeheader()
            writer.writerows(results)

        print(f"Selesai: {doc_subdir.name} → {output_csv}")


if __name__ == "__main__":
    process_all(
        doc_dir=r"C:\Users\Liza\Documents\Kerja Praktik\dokumen\idn",
        json_dir=r"C:\Users\Liza\Documents\Kerja Praktik\dataset\idn",
        output_score=r"C:\Users\Liza\Documents\Kerja Praktik\NLI\Hasil score\idn2"
    )



=== Memproses direktori: D13-Brain Power ===


Proses D13-Brain Power: 100%|██████████| 12/12 [05:38<00:00, 28.19s/it]


Selesai: D13-Brain Power → C:\Users\Liza\Documents\Kerja Praktik\NLI\Hasil score\idn2\D13-Brain Power.csv

=== Memproses direktori: D14-EmotionalFirstAid ===


Proses D14-EmotionalFirstAid: 100%|██████████| 2/2 [01:48<00:00, 54.48s/it]


Selesai: D14-EmotionalFirstAid → C:\Users\Liza\Documents\Kerja Praktik\NLI\Hasil score\idn2\D14-EmotionalFirstAid.csv

=== Memproses direktori: D15-PsikologiKepribadian ===


...

Selesai: D4-Pengantar Psikologi Umum (BimoW) → C:\Users\Liza\Documents\Kerja Praktik\NLI\Hasil score\idn2\D4-Pengantar Psikologi Umum (BimoW).csv

=== Memproses direktori: D5-MindfullLife ===


Proses D5-MindfullLife: 100%|██████████| 2/2 [00:42<00:00, 21.01s/it]


Selesai: D5-MindfullLife → C:\Users\Liza\Documents\Kerja Praktik\NLI\Hasil score\idn2\D5-MindfullLife.csv

=== Memproses direktori: D7-Seni Menyiksa Diri ===


Proses D7-Seni Menyiksa Diri: 100%|██████████| 6/6 [00:46<00:00,  7.67s/it]

Selesai: D7-Seni Menyiksa Diri → C:\Users\Liza\Documents\Kerja Praktik\NLI\Hasil score\idn2\D7-Seni Menyiksa Diri.csv


## IDN OCR

In [1]:
import os, re, json, math, pdfplumber, torch, csv
import pytesseract
from transformers import AutoTokenizer, AutoModelForSequenceClassification

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

MODEL_NAME = "facebook/bart-large-mnli"
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(device)
label2id = {v.lower(): k for k, v in model.config.id2label.items()}

In [2]:
import re
import csv
import json
import torch
import pytesseract
from tqdm import tqdm
from pathlib import Path
from pdf2image import convert_from_path

def extract_text_from_scanned_pdf(pdf_path: str) -> str:
    try:
        images = convert_from_path(pdf_path, dpi=300)
        text_list = [pytesseract.image_to_string(img, lang="ind") for img in images]
        return re.sub(r"\s+", " ", "\n".join(text_list)).strip()
    except Exception as e:
        print(f"OCR gagal untuk {pdf_path}: {e}")
        return ""

def chunk_text(txt: str, max_tokens: int = 256, overlap: int = 64):
    words = txt.split()
    step = max_tokens - overlap
    for i in range(0, len(words), step):
        yield " ".join(words[i:i + max_tokens])

@torch.inference_mode()
def nli_score(premise_chunks, hypothesis: str):
    max_scores = {"entailment": 0.0, "neutral": 0.0, "contradiction": 0.0}
    for prem in premise_chunks:
        enc = tokenizer(prem, hypothesis, truncation=True, return_tensors="pt").to(device)
        logits = model(**enc).logits
        probs = torch.softmax(logits, -1)[0]
        for label in max_scores:
            max_scores[label] = max(max_scores[label], probs[label2id[label]].item())
    return max_scores


def extract_range(filename):
    match = re.search(r"(\d{1,3}-\d{1,3})", filename)
    return match.group(1) if match else None


# === Main Otomatisasi Semua Folder ===
def process_all(main_doc_dir: str, main_json_dir: str, output_root: str):
    main_doc_dir = Path(main_doc_dir)
    main_json_dir = Path(main_json_dir)
    output_root = Path(output_root)
    output_root.mkdir(parents=True, exist_ok=True)

    # Loop per subdirektori dokumen
    for doc_subdir in sorted(main_doc_dir.iterdir()):
        if not doc_subdir.is_dir():
            continue

        dir_prefix = doc_subdir.name.split("-")[0]  # contoh: D3
        json_subdir = next((d for d in main_json_dir.iterdir() if d.name.startswith(dir_prefix)), None)

        if not json_subdir:
            print(f"Skip: Tidak ditemukan pasangan folder dataset untuk {doc_subdir.name}")
            continue

        print(f"\n=== Memproses direktori: {doc_subdir.name} ===")

        results = []
        pdf_files = list(doc_subdir.glob("*.pdf"))
        json_files = list(json_subdir.glob("*.json"))

        json_map = {extract_range(f.name): f for f in json_files if extract_range(f.name)}

        for pdf_path in tqdm(pdf_files, desc=f"Proses {doc_subdir.name}"):
            range_key = extract_range(pdf_path.name)
            if not range_key or range_key not in json_map:
                print(f"Skip: {pdf_path.name} tidak ada pasangan JSON cocok.")
                continue

            json_path = json_map[range_key]
            try:
                teks_ocr = extract_text_from_scanned_pdf(str(pdf_path))
                premise_chunks = list(chunk_text(teks_ocr))
                qna_data = json.loads(json_path.read_text(encoding="utf-8"))
                source = "grok" if "grok" in json_path.name.lower() else "gpt"

                for item in qna_data:
                    question = item.get("Question", "").strip()
                    answer = item.get("Answer", "").strip()
                    hypothesis = answer or f"{question} {answer}".strip()
                    scores = nli_score(premise_chunks, hypothesis)

                    results.append({
                        "file": pdf_path.stem,
                        "generator": source,
                        "question": question,
                        "answer": answer,
                        "score_entailment": round(scores["entailment"], 4),
                        "score_neutral": round(scores["neutral"], 4),
                        "score_contradiction": round(scores["contradiction"], 4),
                        "score_final": round(scores["entailment"] - scores["contradiction"], 4)
                    })
            except Exception as e:
                print(f"Gagal memproses {pdf_path.name}: {e}")

        output_csv = output_root / f"{doc_subdir.name}.csv"
        with open(output_csv, "w", encoding="utf-8", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=[
                "file", "generator", "question", "answer",
                "score_entailment", "score_neutral", "score_contradiction", "score_final"
            ], delimiter=";")
            writer.writeheader()
            writer.writerows(results)

        print(f"Selesai: {doc_subdir.name} → {output_csv}")


if __name__ == "__main__":
    pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
    process_all(
        main_doc_dir=r"C:\Users\Liza\Documents\Kerja Praktik\dokumen\idn-ocr",
        main_json_dir=r"C:\Users\Liza\Documents\Kerja Praktik\dataset\idn-ocr",
        output_root=r"C:\Users\Liza\Documents\Kerja Praktik\NLI\Hasil score\idn-ocr2"
    )



=== Memproses direktori: D10-Hidup Damai Tanpa Berpikir Berlebihan ===


Proses D10-Hidup Damai Tanpa Berpikir Berlebihan:   0%|          | 0/13 [00:00<?, ?it/s]Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.58.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.
Proses D10-Hidup Damai Tanpa Berpikir Berlebihan: 100%|██████████| 13/13 [09:30<00:00, 43.86s/it]


Selesai: D10-Hidup Damai Tanpa Berpikir Berlebihan → C:\Users\Liza\Documents\Kerja Praktik\NLI\Hasil score\idn-ocr2\D10-Hidup Damai Tanpa Berpikir Berlebihan.csv

=== Memproses direktori: D6-Berani tidak disukai x Ichiro Kishimi ===


Proses D6-Berani tidak disukai x Ichiro Kishimi: 100%|██████████| 30/30 [12:51<00:00, 25.73s/it]


Selesai: D6-Berani tidak disukai x Ichiro Kishimi → C:\Users\Liza\Documents\Kerja Praktik\NLI\Hasil score\idn-ocr2\D6-Berani tidak disukai x Ichiro Kishimi.csv

=== Memproses direktori: D8-Berdamai Dengan Diri Sendiri ===


Proses D8-Berdamai Dengan Diri Sendiri: 100%|██████████| 19/19 [07:47<00:00, 24.61s/it]


Selesai: D8-Berdamai Dengan Diri Sendiri → C:\Users\Liza\Documents\Kerja Praktik\NLI\Hasil score\idn-ocr2\D8-Berdamai Dengan Diri Sendiri.csv

=== Memproses direktori: D9-EgoIsTheEnemy ===


Proses D9-EgoIsTheEnemy: 100%|██████████| 24/24 [14:19<00:00, 35.81s/it]

Selesai: D9-EgoIsTheEnemy → C:\Users\Liza\Documents\Kerja Praktik\NLI\Hasil score\idn-ocr2\D9-EgoIsTheEnemy.csv


## MARIAN ENG

In [1]:
import os, re, json, csv, math, torch, pytesseract, nltk
from tqdm import tqdm
from pathlib import Path
from pdf2image import convert_from_path
from nltk.tokenize import sent_tokenize
from transformers import (MarianTokenizer, MarianMTModel, AutoTokenizer, AutoModelForSequenceClassification)

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
nltk.download('punkt')
nltk.download('punkt_tab')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# === Load Translation Model EN-ID ===
trans_model_name = "Helsinki-NLP/opus-mt-en-id"
trans_tokenizer = MarianTokenizer.from_pretrained(trans_model_name)
trans_model = MarianMTModel.from_pretrained(trans_model_name).to(device)

# === Load NLI Model ===
nli_model_name = "facebook/bart-large-mnli"
nli_tokenizer = AutoTokenizer.from_pretrained(nli_model_name)
nli_model = AutoModelForSequenceClassification.from_pretrained(nli_model_name).to(device)
label2id = {v.lower(): k for k, v in nli_model.config.id2label.items()}


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Liza\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Liza\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
c:\Users\Liza\anaconda3\envs\panda\Lib\site-packages\transformers\models\marian\tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [2]:
import os
import re
import csv
import json
import torch
from tqdm import tqdm
from pathlib import Path
from pdf2image import convert_from_path
import pytesseract
from nltk.tokenize import sent_tokenize

# === OCR (ENG) ===
def extract_text_from_scanned_pdf(pdf_path: str) -> str:
    print(f"OCR dari: {pdf_path}")
    images = convert_from_path(pdf_path, dpi=300)
    text_list = [pytesseract.image_to_string(img, lang="eng") for img in images]
    return re.sub(r"\s+", " ", "\n".join(text_list)).strip()


# === Translate ENG -> IDN ===
def translate_text(text: str) -> str:
    sentences = sent_tokenize(text)
    translated_sentences = []

    for sentence in tqdm(sentences, desc="Translasi"):
        inputs = trans_tokenizer(sentence, return_tensors="pt", truncation=True, padding=True).to(device)
        outputs = trans_model.generate(**inputs, max_length=512)
        translated = trans_tokenizer.decode(outputs[0], skip_special_tokens=True)
        translated_sentences.append(translated)

    return " ".join(translated_sentences)


# === Potong teks ===
def chunk_text(txt: str, max_tokens: int = 256, overlap: int = 64):
    words = txt.split()
    step = max_tokens - overlap
    for i in range(0, len(words), step):
        yield " ".join(words[i:i + max_tokens])


# === Hitung skor NLI (entailment, neutral, contradiction) ===
@torch.inference_mode()
def nli_score(premise_chunks, hypothesis: str):
    max_scores = {"entailment": 0.0, "neutral": 0.0, "contradiction": 0.0}
    for prem in premise_chunks:
        enc = nli_tokenizer(prem, hypothesis, truncation=True, return_tensors="pt").to(device)
        logits = nli_model(**enc).logits
        probs = torch.softmax(logits, -1)[0]
        for label in max_scores:
            max_scores[label] = max(max_scores[label], probs[label2id[label]].item())
    return {k: round(v, 4) for k, v in max_scores.items()}


def process_all(base_pdf_dir: str, base_json_dir: str, output_dir: str):
    os.makedirs(output_dir, exist_ok=True)

    doc_dirs = [d for d in Path(base_pdf_dir).iterdir() if d.is_dir()]

    for doc_dir in doc_dirs:
        dir_name = doc_dir.name
        json_dir = Path(base_json_dir) / dir_name
        if not json_dir.exists():
            print(f"Tidak ditemukan pasangan dataset untuk {dir_name}")
            continue

        print(f"\n=== Evaluasi direktori: {dir_name} ===")
        results = []

        pdf_files = list(doc_dir.glob("*.pdf"))
        json_files = list(json_dir.glob("*.json"))

        def extract_range(filename):
            match = re.search(r"(\d{1,3}-\d{1,3})", filename)
            return match.group(1) if match else None

        json_map = {extract_range(f.name): f for f in json_files if extract_range(f.name)}

        with tqdm(total=len(pdf_files), desc=f"Evaluasi {dir_name}") as pbar:
            for pdf_path in pdf_files:
                range_key = extract_range(pdf_path.name)
                if not range_key or range_key not in json_map:
                    tqdm.write(f"Tidak menemukan JSON untuk {pdf_path.name}")
                    pbar.update(1)
                    continue

                json_path = json_map[range_key]
                source = "grok" if "grok" in json_path.name.lower() else "gpt"
                tqdm.write(f"\nProses: {pdf_path.name} + {json_path.name}")

                # === OCR + Translate ===
                ocr_text = extract_text_from_scanned_pdf(str(pdf_path))
                id_text = translate_text(ocr_text)
                chunks = list(chunk_text(id_text))

                # === Load dataset JSON ===
                with open(json_path, "r", encoding="utf-8") as f:
                    qna_data = json.load(f)

                # === Evaluasi tiap QnA ===
                for item in qna_data:
                    question = item.get("Question", "").strip()
                    answer = item.get("Answer", "").strip()
                    hypothesis = answer or f"{question} {answer}".strip()

                    scores = nli_score(chunks, hypothesis)

                    results.append({
                        "file": pdf_path.stem,
                        "generator": source,
                        "question": question,
                        "answer": answer,
                        "score_entailment": scores["entailment"],
                        "score_neutral": scores["neutral"],
                        "score_contradiction": scores["contradiction"],
                        "score_final": round(scores["entailment"] - scores["contradiction"], 4)
                    })

                pbar.update(1)

        output_csv = Path(output_dir) / f"{dir_name}.csv"
        with open(output_csv, "w", encoding="utf-8", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=[
               "file", "generator", "question", "answer",
                "score_entailment", "score_neutral", "score_contradiction", "score_final"
            ], delimiter=";")
            writer.writeheader()
            writer.writerows(results)

        print(f"Selesai: {output_csv}")


if __name__ == "__main__":
    process_all(
        base_pdf_dir=r"C:\Users\Liza\Documents\Kerja Praktik\dokumen\eng",
        base_json_dir=r"C:\Users\Liza\Documents\Kerja Praktik\dataset\eng",
        output_dir=r"C:\Users\Liza\Documents\Kerja Praktik\NLI\Hasil score\eng2"
    )



=== Evaluasi direktori: D1-Health Psychology, 9th Ed (Shelley E. Taylor) ===


Evaluasi D1-Health Psychology, 9th Ed (Shelley E. Taylor):   0%|          | 0/10 [00:00<?, ?it/s]


Proses: Health Psychology, 9th Ed (Shelley E. Taylor)-1-55.pdf + D1_GROK_HealthPsycology1-55_67.json
OCR dari: C:\Users\Liza\Documents\Kerja Praktik\dokumen\eng\D1-Health Psychology, 9th Ed (Shelley E. Taylor)\Health Psychology, 9th Ed (Shelley E. Taylor)-1-55.pdf


Translasi: 100%|██████████| 1118/1118 [04:44<00:00,  3.93it/s]
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.58.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.
Evaluasi D1-Health Psychology, 9th Ed (Shelley E. Taylor):  10%|█         | 1/10 [23:06<3:28:00, 1386.69s/it]


Proses: Health Psychology, 9th Ed (Shelley E. Taylor)-114-151.pdf + D1_GROK_HealthPsycology114-151_58.json
OCR dari: C:\Users\Liza\Documents\Kerja Praktik\dokumen\eng\D1-Health Psychology, 9th Ed (Shelley E. Taylor)\Health Psychology, 9th Ed (Shelley E. Taylor)-114-151.pdf


...


Proses: A Handbook for the Study of Mental Health Social Contexts, Theories, and Systems (Teresa L. Scheid, Tony N. Brown) (z-lib.org)-380-390.pdf + D11_GROK_AHandbookfortheSMH380-390_20.json
OCR dari: C:\Users\Liza\Documents\Kerja Praktik\dokumen\eng\D11-A Handbook for the Study of Mental Health Social\A Handbook for the Study of Mental Health Social Contexts, Theories, and Systems (Teresa L. Scheid, Tony N. Brown) (z-lib.org)-380-390.pdf


Translasi: 100%|██████████| 240/240 [00:49<00:00,  4.81it/s]
Evaluasi D11-A Handbook for the Study of Mental Health Social:  52%|█████▏    | 28/54 [54:52<52:02, 120.08s/it]


Proses: A Handbook for the Study of Mental Health Social Contexts, Theories, and Systems (Teresa L. Scheid, Tony N. Brown) (z-lib.org)-39-49.pdf + D11_GROK_AHandbookfortheSMH39-49_20.json
OCR dari: C:\Users\Liza\Documents\Kerja Praktik\dokumen\eng\D11-A Handbook for the Study of Mental Health Social\A Handbook for the Study of Mental Health Social Contexts, Theories, and Systems (Teresa L. Scheid, Tony N. Brown) (z-lib.org)-39-49.pdf


Translasi: 100%|██████████| 163/163 [00:44<00:00,  3.70it/s]
Evaluasi D11-A Handbook for the Study of Mental Health Social:  52%|█████▏    | 28/54 [56:08<52:07, 120.29s/it]


KeyboardInterrupt: 

In [6]:
import os
import pandas as pd

base_dir = r"C:\Users\Liza\Documents\Kerja Praktik\NLI\Hasil score"
folders = ["eng", "idn", "idn-ocr"]

merged_dfs = {}

def safe_read_csv(csv_file, expected_cols=None):
    """
    Membaca CSV dengan berbagai fallback agar tetap bisa diparse
    meskipun formatnya tidak konsisten.
    """
    df = None
    try:
        # Coba normal dulu (auto-deteksi delimiter)
        df = pd.read_csv(csv_file, sep=None, engine="python", encoding="utf-8-sig", quotechar='"')
    except Exception as e1:
        print(f"Error baca {csv_file}: {e1}")

        try:
            # Coba pakai koma eksplisit
            df = pd.read_csv(csv_file, sep=",", quotechar='"', engine="python", encoding="utf-8-sig")
        except Exception as e2:
            print(f"Fallback kedua gagal: {e2}")
            try:
                # Fallback terakhir: parsing manual per baris
                with open(csv_file, "r", encoding="utf-8-sig", errors="replace") as f:
                    lines = [line.strip() for line in f if line.strip()]
                rows = []
                for line in lines:
                    parts = []
                    current = ""
                    in_quotes = False
                    for char in line:
                        if char == '"':
                            in_quotes = not in_quotes
                        elif char == "," and not in_quotes:
                            parts.append(current)
                            current = ""
                        else:
                            current += char
                    parts.append(current)
                    rows.append(parts)

                max_cols = max(len(r) for r in rows)
                df = pd.DataFrame(rows, columns=[f"col_{i}" for i in range(max_cols)])
                print(f"Dibaca pakai parser manual: {csv_file} ({max_cols} kolom)")
            except Exception as e3:
                print(f"Gagal total baca {csv_file}: {e3}")
                return pd.DataFrame()

    # Samakan jumlah kolom antar file
    if expected_cols is not None:
        for col in expected_cols:
            if col not in df.columns:
                df[col] = None
        df = df[expected_cols]

    return df


for folder in folders:
    folder_path = os.path.join(base_dir, folder)
    all_csv_files = []

    for root, dirs, files in os.walk(folder_path):
        for file in files:
            if file.endswith(".csv"):
                all_csv_files.append(os.path.join(root, file))

    if not all_csv_files:
        print(f"Tidak ada file CSV di {folder}")
        continue

    print(f"\nMemproses folder: {folder}")
    first_df = safe_read_csv(all_csv_files[0])
    expected_cols = list(first_df.columns)
    dfs = [first_df]

    for csv_file in all_csv_files[1:]:
        df = safe_read_csv(csv_file, expected_cols)
        dfs.append(df)

    combined_df = pd.concat(dfs, ignore_index=True)
    combined_df = combined_df.dropna(how="all")  # hapus baris kosong

    output_path = os.path.join(base_dir, f"{folder}_all.csv")
    combined_df.to_csv(output_path, index=False, encoding="utf-8-sig")

    merged_dfs[folder] = combined_df
    print(f"{folder}: {len(dfs)} file digabung → {output_path}")

# Gabungkan semua folder
if merged_dfs:
    final_df = pd.concat(merged_dfs.values(), ignore_index=True, sort=False)
    final_path = os.path.join(base_dir, "final_score_NLI.csv")
    final_df.to_csv(final_path, index=False, encoding="utf-8-sig")
    print(f"\nSemua selesai. File akhir disimpan di:\n{final_path}")
else:
    print("Tidak ada file CSV valid yang ditemukan untuk digabungkan.")



Memproses folder: eng
eng: 4 file digabung → C:\Users\Liza\Documents\Kerja Praktik\NLI\Hasil score\eng_all.csv

Memproses folder: idn
Error baca C:\Users\Liza\Documents\Kerja Praktik\NLI\Hasil score\idn\D15-PsikologiKepribadian.csv: ',' expected after '"'
Fallback kedua gagal: ',' expected after '"'
Dibaca pakai parser manual: C:\Users\Liza\Documents\Kerja Praktik\NLI\Hasil score\idn\D15-PsikologiKepribadian.csv (8 kolom)
idn: 10 file digabung → C:\Users\Liza\Documents\Kerja Praktik\NLI\Hasil score\idn_all.csv

Memproses folder: idn-ocr
idn-ocr: 4 file digabung → C:\Users\Liza\Documents\Kerja Praktik\NLI\Hasil score\idn-ocr_all.csv

Semua selesai. File akhir disimpan di:
C:\Users\Liza\Documents\Kerja Praktik\NLI\Hasil score\final_score_NLI.csv


C:\Users\Liza\AppData\Local\Temp\ipykernel_11864\744678218.py:85: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined_df = pd.concat(dfs, ignore_index=True)


In [14]:
import pandas as pd

# Baca ulang CSV dengan delimiter yang benar
df = pd.read_csv("Hasil score/idn-ocr_all.csv")
df



,file,generator,question,answer,score_entailment,score_neutral,score_contradiction,score_final
0,Hidup Damai Tanpa Berpikir Berlebihan x Tsunek...,grok,Aku merasa tertekan banget kalau kerjaan menum...,Wajar kok merasa tertekan kalau kerjaan numpuk...,0.8699,0.2814,0.2496,0.6203
1,Hidup Damai Tanpa Berpikir Berlebihan x Tsunek...,grok,Kadang aku ragu bisa nggak ya ngerjain tugas y...,"Ragu itu manusiawi banget. Coba aja dulu, angg...",0.7990,0.2718,0.4013,0.3977
2,Hidup Damai Tanpa Berpikir Berlebihan x Tsunek...,grok,"Aku takut gagal kalau coba sesuatu yang baru, ...","Takut gagal itu normal, kok. Tapi coba pikir, ...",0.8063,0.3049,0.4347,0.3715
3,Hidup Damai Tanpa Berpikir Berlebihan x Tsunek...,grok,"Aku sering kurang tidur karena kerja, ini buru...","Wah, kurang tidur itu bisa bikin tubuh dan pik...",0.7062,0.2959,0.4438,0.2623
4,Hidup Damai Tanpa Berpikir Berlebihan x Tsunek...,grok,Aku pengen kerja keras terus biar dianggap heb...,"Nggak salah pengen berprestasi, tapi kalau sam...",0.7865,0.2849,0.4542,0.3324
...,...,...,...,...,...,...,...,...
1524,Ego is the Enemy - Ryan Holiday-94-105,grok,Aku kesal banget sama rekan kerja yang selalu ...,"Kesal itu wajar, tapi kasih pelajaran langsung...",0.9044,0.2464,0.6489,0.2556
1525,Ego is the Enemy - Ryan Holiday-94-105,grok,Aku merasa nggak cukup baik dibandingkan orang...,"Perasaan itu sering datang, tapi coba ingat: k...",0.8270,0.3752,0.4388,0.3882
1526,Ego is the Enemy - Ryan Holiday-94-105,grok,Aku pengen ngeluh terus kalau kerjaan nggak ad...,"Ngeluh sesekali boleh, tapi kalau terus-terusa...",0.8327,0.2022,0.6467,0.1860
1527,Ego is the Enemy - Ryan Holiday-94-105,grok,Aku merasa orang lain nggak pantas dapat posis...,"Wajar kalau merasa kecewa, tapi fokus ke orang...",0.9241,0.2168,0.6458,0.2783


In [15]:
df.isna().any()

file                   False
generator              False
question                True
answer                  True
score_entailment       False
score_neutral          False
score_contradiction    False
score_final            False
dtype: bool

In [16]:
df[df['question'].isna() | df['answer'].isna()]


,file,generator,question,answer,score_entailment,score_neutral,score_contradiction,score_final
540,Berani tidak disukai x Ichiro Kishimi Dan Fumi...,grok,NaN,NaN,0.8643,0.4493,0.8615,0.0028
541,Berani tidak disukai x Ichiro Kishimi Dan Fumi...,grok,NaN,NaN,0.8643,0.4493,0.8615,0.0028
542,Berani tidak disukai x Ichiro Kishimi Dan Fumi...,grok,NaN,NaN,0.8643,0.4493,0.8615,0.0028
543,Berani tidak disukai x Ichiro Kishimi Dan Fumi...,grok,NaN,NaN,0.8643,0.4493,0.8615,0.0028
544,Berani tidak disukai x Ichiro Kishimi Dan Fumi...,grok,NaN,NaN,0.8643,0.4493,0.8615,0.0028
545,Berani tidak disukai x Ichiro Kishimi Dan Fumi...,grok,NaN,NaN,0.8643,0.4493,0.8615,0.0028
546,Berani tidak disukai x Ichiro Kishimi Dan Fumi...,grok,NaN,NaN,0.8643,0.4493,0.8615,0.0028
547,Berani tidak disukai x Ichiro Kishimi Dan Fumi...,grok,NaN,NaN,0.8643,0.4493,0.8615,0.0028
548,Berani tidak disukai x Ichiro Kishimi Dan Fumi...,grok,NaN,NaN,0.8643,0.4493,0.8615,0.0028
549,Berani tidak disukai x Ichiro Kishimi Dan Fumi...,grok,NaN,NaN,0.8643,0.4493,0.8615,0.0028


In [21]:
df.describe(include='all')

,file,generator,question,answer,entailment,neutral,contradiction,score
count,3443,3443,3443,3443,3443.000000,3443.000000,3443.000000,3443.000000
unique,144,2,3438,3443,NaN,NaN,NaN,NaN
top,"Health Psychology, 9th Ed (Shelley E. Taylor)-...",grok,Aku ngerasa hidupku nggak ada artinya. Apa aku...,"Tentu, saya paham kalau istilah ini bisa teras...",NaN,NaN,NaN,NaN
freq,70,3151,2,1,NaN,NaN,NaN,NaN
mean,NaN,NaN,NaN,NaN,0.867640,0.262558,0.466867,0.400773
std,NaN,NaN,NaN,NaN,0.070872,0.087365,0.149960,0.190984
min,NaN,NaN,NaN,NaN,0.383000,0.073400,0.085500,-0.357100
25%,NaN,NaN,NaN,NaN,0.829400,0.206250,0.358100,0.274800
50%,NaN,NaN,NaN,NaN,0.879600,0.246500,0.455700,0.413700
75%,NaN,NaN,NaN,NaN,0.919100,0.296500,0.569450,0.538350


In [6]:
import pandas as pd

df = pd.read_csv(r"Hasil score\score_nli_result.csv", sep=';')
df


,file,generator,question,answer,score_entailment,score_neutral,score_contradiction,score_final
0,"Health Psychology, 9th Ed (Shelley E. Taylor)-...",grok,Saya merasa bingung tentang apa itu kesehatan ...,"Tentu, saya paham kalau istilah ini bisa teras...",0.9724,0.2708,0.2997,0.6727
1,"Health Psychology, 9th Ed (Shelley E. Taylor)-...",grok,Saya sering merasa kesehatan saya dipengaruhi ...,Keren banget kamu peka sama tubuh dan pikiranm...,0.9827,0.2300,0.3361,0.6466
2,"Health Psychology, 9th Ed (Shelley E. Taylor)-...",grok,"Akhir-akhir ini saya merasa stres berat, dan t...","Wah, saya tahu rasanya saat stres bikin semuan...",0.9030,0.2509,0.6111,0.2919
3,"Health Psychology, 9th Ed (Shelley E. Taylor)-...",grok,"Saya merasa sendiri akhir-akhir ini, dan itu b...","Aduh, rasanya pasti nggak enak kalau merasa se...",0.9556,0.3639,0.5771,0.3785
4,"Health Psychology, 9th Ed (Shelley E. Taylor)-...",grok,Saya sering merasa kewalahan saat menghadapi m...,"Saya paham, kadang masalah terasa seperti gunu...",0.9218,0.2387,0.5177,0.4041
...,...,...,...,...,...,...,...,...
7495,Ego is the Enemy - Ryan Holiday-94-105,grok,Aku kesal banget sama rekan kerja yang selalu ...,"Kesal itu wajar, tapi kasih pelajaran langsung...",0.9044,0.2464,0.6489,0.2556
7496,Ego is the Enemy - Ryan Holiday-94-105,grok,Aku merasa nggak cukup baik dibandingkan orang...,"Perasaan itu sering datang, tapi coba ingat: k...",0.8270,0.3752,0.4388,0.3882
7497,Ego is the Enemy - Ryan Holiday-94-105,grok,Aku pengen ngeluh terus kalau kerjaan nggak ad...,"Ngeluh sesekali boleh, tapi kalau terus-terusa...",0.8327,0.2022,0.6467,0.1860
7498,Ego is the Enemy - Ryan Holiday-94-105,grok,Aku merasa orang lain nggak pantas dapat posis...,"Wajar kalau merasa kecewa, tapi fokus ke orang...",0.9241,0.2168,0.6458,0.2783


In [8]:
df.describe()

,score_entailment,score_neutral,score_contradiction,score_final
count,7500.000000,7500.000000,7500.000000,7500.000000
mean,0.847825,0.247116,0.460885,0.386940
std,0.086240,0.085581,0.157506,0.202914
min,0.282300,0.059500,0.036300,-0.699800
25%,0.801700,0.192975,0.344600,0.253000
50%,0.865000,0.233500,0.450550,0.401750
75%,0.911525,0.282525,0.570100,0.530825
max,0.989300,0.925800,0.982000,0.922500
